# Observability: Metrics, Logs, and Harness Traces

This notebook covers the observability stack for the Custom Deep Research Lab.
You will verify the platform components (Grafana, LokiStack, Prometheus), review
agentic metrics, write LogQL queries, and query the harness `trace_events` table
for per-session metrics.

## 1. Prerequisites

In [1]:
import os
import subprocess
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

ns = os.getenv("NAMESPACE", "doc-research-lab")
result = subprocess.run(["oc", "get", "namespace", ns], capture_output=True, text=True)
if result.returncode == 0:
    print(f"\u2705 Namespace '{ns}' exists")
else:
    print(f"\u274c Namespace '{ns}' not found \u2014 deploy the application first")

✅ Namespace 'doc-research-lab' exists


## 2. Verify Observability Stack

Check that LokiStack, the ServiceMonitor, and the GrafanaDashboard CR are deployed.

In [2]:
checks = [
    ("LokiStack", "oc get lokistack -A -o name"),
    ("ServiceMonitor", f"oc get servicemonitor research-backend -n {ns} -o name"),
    ("GrafanaDashboard", f"oc get grafanadashboard research-lab-metrics -n {ns} -o name"),
]
for name, cmd in checks:
    result = subprocess.run(cmd.split(), capture_output=True, text=True)
    status = "\u2705" if result.returncode == 0 else "\u26a0\ufe0f Not found"
    print(f"  {status} {name}")

  ⚠️ Not found LokiStack
  ⚠️ Not found ServiceMonitor
  ⚠️ Not found GrafanaDashboard


## 3. Key Agentic Metrics (Prometheus)

The research backend exposes Prometheus metrics on `/metrics`. These are scraped by the
`ServiceMonitor` and visualized in the Grafana dashboard.

| # | Metric | Prometheus Name | Description |
|---|--------|----------------|-------------|
| 1 | Request Rate | `research_requests_total` | Total number of research requests received |
| 2 | Latency Percentiles | `research_request_duration_seconds` | End-to-end request latency (p50, p90, p99) |
| 3 | LLM Token Usage | `llm_tokens_total` | Token consumption by model and type (input/output) |
| 4 | LLM Inference Latency | `llm_inference_duration_seconds` | Time spent in LLM inference calls |
| 5 | Tool Call Success Rate | `tool_calls_total` | MCP tool invocations by name and status (success/error) |
| 6 | Research Quality Scores | `research_quality_score` | LLM-as-Judge quality scores per iteration |

These metrics are sufficient to monitor the health and efficiency of any agentic application.

## 4. Explore the Grafana Dashboard

The `research-lab-metrics` GrafanaDashboard CR deploys a pre-built dashboard.

**How to access:**
1. Open the OpenShift Console
2. Navigate to **Networking \u2192 Routes** in the `doc-research-lab` namespace
3. Click the Grafana route URL
4. Log in with your OpenShift credentials (OAuth proxy)
5. Select the **Research Lab Metrics** dashboard

**Dashboard panels:**

| Row | Panel | Metrics Used |
|-----|-------|---------- ----|
| Overview | Request Rate (req/min) | `rate(research_requests_total[5m])` |
| Overview | Latency Percentiles | `histogram_quantile(0.95, research_request_duration_seconds_bucket)` |
| Overview | Active Requests | `research_active_requests` |
| AI Agent | Token Usage by Model | `sum by(model)(rate(llm_tokens_total[5m]))` |
| AI Agent | LLM Inference Latency | `histogram_quantile(0.95, llm_inference_duration_seconds_bucket)` |
| AI Agent | Tool Call Success Rate | `sum(rate(tool_calls_total{status="success"}[5m]))` |
| AI Agent | Quality Scores Over Time | `research_quality_score` |

## 5. LogQL Queries

LokiStack automatically collects STDOUT and STDERR from all containers running in the
cluster. The research backend uses structured JSON logging (`harness/logging.py`), so
you can parse and filter fields directly in LogQL.

**Where to run these queries:**
- OpenShift Console \u2192 **Observe \u2192 Logs**
- Or the Grafana Explore tab with the Loki data source

In [7]:
queries = [
    ("All application logs",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }}'),
    ("Backend logs only",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}", kubernetes_container_name="backend" }} | json | line_format "{{{{.message}}}}"'),
    ("Error logs",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | level="ERROR"'),
    ("Verification results",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | layer="verification"'),
    ("Search operations",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | operation=~"search.*"'),
    ("Report generation rate",
     f'rate({{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | operation="draft_report" [5m])'),
]

print("LogQL Queries for OpenShift Console \u2192 Observe \u2192 Logs:\n")
for desc, q in queries:
    print(f"\U0001f4cb {desc}:")
    print(f"   {q}\n")
print("\u2705 Copy these queries into the OpenShift Console log viewer")

LogQL Queries for OpenShift Console → Observe → Logs:

📋 All application logs:
   { log_type="application", kubernetes_namespace_name="doc-research-lab" }

📋 Backend logs only:
   { log_type="application", kubernetes_namespace_name="doc-research-lab", kubernetes_container_name="backend" } | json | line_format "{{.message}}"

📋 Error logs:
   { log_type="application", kubernetes_namespace_name="doc-research-lab" } | json | level="ERROR"

📋 Verification results:
   { log_type="application", kubernetes_namespace_name="doc-research-lab" } | json | layer="verification"

📋 Search operations:
   { log_type="application", kubernetes_namespace_name="doc-research-lab" } | json | operation=~"search.*"

📋 Report generation rate:
   rate({ log_type="application", kubernetes_namespace_name="doc-research-lab" } | json | operation="draft_report" [5m])

✅ Copy these queries into the OpenShift Console log viewer


## 6. Harness Trace Events (PostgreSQL)

The harness records fine-grained trace events to a `trace_events` table in PostgreSQL.
Each event captures session ID, iteration, layer, operation, latency, token usage, and
failure categories. This complements Prometheus metrics with per-session detail.

In [8]:
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))
from db import get_connection

try:
    conn = get_connection()
    row = conn.execute(
        "SELECT tablename FROM pg_tables WHERE schemaname = 'public' AND tablename = %s",
        ("trace_events",),
    ).fetchone()
    if not row:
        print("\u26a0\ufe0f trace_events table not found \u2014 run a research query first")
        conn.close()
        conn = None
    else:
        print("\u2705 Connected to PostgreSQL, trace_events table found")
except Exception as e:
    print(f"\u26a0\ufe0f PostgreSQL unavailable ({e})")
    conn = None

✅ Connected to PostgreSQL, trace_events table found


Query per-session metrics from the `trace_events` table.

In [9]:
if conn:
    rows = conn.execute("""
        SELECT
            session_id,
            COUNT(*) AS total_events,
            SUM(tokens_used) AS total_tokens,
            SUM(latency_ms) AS total_latency_ms,
            SUM(CASE WHEN NOT success THEN 1 ELSE 0 END) AS failures,
            MAX(iteration) AS max_iteration
        FROM trace_events
        GROUP BY session_id
        ORDER BY MAX(timestamp) DESC
        LIMIT 10
    """).fetchall()
    if rows:
        print(f"{'Session':>12} {'Events':>7} {'Tokens':>8} {'Latency':>10} {'Fails':>6} {'Iters':>6}")
        print("-" * 55)
        for r in rows:
            print(f"{r['session_id']:>12} {r['total_events']:>7} {r['total_tokens']:>8} {r['total_latency_ms']:>8}ms {r['failures']:>6} {r['max_iteration']:>6}")
    else:
        print("No trace events found \u2014 run a research query first")
    conn.close()
    print("\n\u2705 Harness trace query complete")
else:
    print("\u26a0\ufe0f Skipped \u2014 PostgreSQL not available")

     Session  Events   Tokens    Latency  Fails  Iters
-------------------------------------------------------
f9b473c9-e47      22    23308        0ms      0      2
d2c4cda4-1bb      19    23199        0ms      0      2
26eade95-422       9     5703        0ms      0      1
cfa3c152-fc8      29    28321        0ms      0      2
e6e3fdcb-64e      49    46624        0ms      0      2
93884837-dda       7     6398        0ms      0      1
502400ef-255       3      471        0ms      0      1
30018db4-bef       3      550        0ms      0      1
1f5bb797-620       7     5810        0ms      0      1
2623ad09-e18      10     6430        0ms      0      1

✅ Harness trace query complete


## 7. Summary

In [10]:
print("\u2705 Observability walkthrough complete")
print()
print("What we covered:")
print("  1. Verified the observability stack (LokiStack, ServiceMonitor, GrafanaDashboard)")
print("  2. Reviewed 6 key agentic Prometheus metrics")
print("  3. Explored the Grafana dashboard panels")
print("  4. Built LogQL queries for application log analysis")
print("  5. Queried harness trace_events for per-session metrics")
print()
print("Next: 3_tracing_mlflow.ipynb \u2014 MLflow tracing for agent spans")

✅ Observability walkthrough complete

What we covered:
  1. Verified the observability stack (LokiStack, ServiceMonitor, GrafanaDashboard)
  2. Reviewed 6 key agentic Prometheus metrics
  3. Explored the Grafana dashboard panels
  4. Built LogQL queries for application log analysis
  5. Queried harness trace_events for per-session metrics

Next: 3_tracing_mlflow.ipynb — MLflow tracing for agent spans


In [ ]:
# End of notebook